# Phase 1 EDNet-KT Smoke Test

MSc Dissertation: A Comparative Study of Knowledge Tracing Models — BKT, DKT and SAKT on EDNet-KT1.

This notebook builds the Phase 1 pipeline from scratch:

1. Set up the environment.
2. Load EDNet-KT1 data.
3. Preprocess interactions and question metadata.
4. Create a small student-level smoke-test subset.
5. Train and evaluate a BKT baseline.
6. Prepare the structure for DKT and SAKT smoke tests.

The purpose of this notebook is not final model performance. It is to verify that the full pipeline works end-to-end.

In [1]:
%pip install -q pyBKT scikit-learn pandas numpy matplotlib


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")


Random seed set to 42


In [38]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"          
CONTENT_DATA_DIR = DATA_DIR / "contents"  

print("Project root:", PROJECT_ROOT)
print("User CSV folder:", RAW_DATA_DIR)
print("Content metadata folder:", CONTENT_DATA_DIR)


Project root: /Users/tolulopesoetan/Desktop/msc_kt_ednet
User CSV folder: /Users/tolulopesoetan/Desktop/msc_kt_ednet/data/raw
Content metadata folder: /Users/tolulopesoetan/Desktop/msc_kt_ednet/data/contents


In [9]:
CONFIG = {
    "seed": SEED,
    "smoke_test_n_students": 100,  # increase to 1000 once the pipeline runs cleanly
    "train_size": 0.8,
    "dataset_name": "EDNet-KT1",
    "user_column": "user_id",
    "question_column": "question_id",
    "target_column": "is_correct",
    "skill_column": "skill_name",
    "min_interactions_per_skill": 10,
}

CONFIG


{'seed': 42,
 'smoke_test_n_students': 100,
 'train_size': 0.8,
 'dataset_name': 'EDNet-KT1',
 'user_column': 'user_id',
 'question_column': 'question_id',
 'target_column': 'is_correct',
 'skill_column': 'skill_name',
 'min_interactions_per_skill': 10}

In [10]:
# Find EDNet-KT1 user CSV files in data/raw.
user_csv_files = sorted([
    p for p in RAW_DATA_DIR.rglob("*.csv")
    if p.name.lower().startswith("u") and p.stem[1:].isdigit()
])

print(f"Found {len(user_csv_files)} user CSV file(s) in data/raw.")

QUESTION_METADATA_PATH = CONTENT_DATA_DIR / "questions.csv"




Found 784309 user CSV file(s) in data/raw.


In [11]:
# Preview one user interaction file.
if len(user_csv_files) == 0:
    raise FileNotFoundError("No user CSV files found. Check that files like u1.csv are inside data/raw.")

sample_user_file = user_csv_files[0]
sample_user_df = pd.read_csv(sample_user_file)

print("Sample user file:", sample_user_file.relative_to(PROJECT_ROOT))
print("Shape:", sample_user_df.shape)
print("Columns:")
print(list(sample_user_df.columns))

display(sample_user_df.head())


Sample user file: data/raw/u1.csv
Shape: (1082, 5)
Columns:
['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time']


,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1565096190868,1,q5012,b,38000
1,1565096221062,2,q4706,c,24000
2,1565096293432,3,q4366,b,68000
3,1565096339668,4,q4829,a,42000
4,1565096401774,5,q6528,b,59000


In [13]:
# Load and preview questions.csv.
if not QUESTION_METADATA_PATH.exists():
    raise FileNotFoundError("questions.csv was not found in data/contents")

questions_df = pd.read_csv(QUESTION_METADATA_PATH)

print("Question metadata shape:", questions_df.shape)
print("Columns:")
print(list(questions_df.columns))

display(questions_df.head())


Question metadata shape: (13169, 7)
Columns:
['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439


In [15]:
# Validate required columns.
required_user_columns = {"timestamp", "solving_id", "question_id", "user_answer", "elapsed_time"}
required_question_columns = {"question_id", "correct_answer", "tags"}

missing_user_columns = required_user_columns - set(sample_user_df.columns)
missing_question_columns = required_question_columns - set(questions_df.columns)

print("Missing columns in sample user CSV:", missing_user_columns if missing_user_columns else "None")
print("Missing columns in questions.csv:", missing_question_columns if missing_question_columns else "None")

if missing_user_columns:
    raise ValueError(f"Sample user CSV is missing required columns: {missing_user_columns}")

if missing_question_columns:
    raise ValueError(f"questions.csv is missing required columns: {missing_question_columns}")

question_skill_count = (
    questions_df["tags"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .nunique()
)

print("Validation passed.")
print(f"  Question rows : {len(questions_df):,}")
print(f"  Unique skills : {question_skill_count:,}")


Missing columns in sample user CSV: None
Missing columns in questions.csv: None
Validation passed.
  Question rows : 13,169
  Unique skills : 189


In [16]:
N_SMOKE_USERS = CONFIG["smoke_test_n_students"]
selected_user_files = user_csv_files[:N_SMOKE_USERS]

print(f"Selected {len(selected_user_files)} user file(s) for this smoke test.")
print("First file:", selected_user_files[0].relative_to(PROJECT_ROOT) if selected_user_files else "None")
print("Last file:", selected_user_files[-1].relative_to(PROJECT_ROOT) if selected_user_files else "None")


Selected 100 user file(s) for this smoke test.
First file: data/raw/u1.csv
Last file: data/raw/u100087.csv


In [17]:
def extract_user_id_from_filename(file_path: Path) -> str:
    """Extract user_id from an EDNet user filename, e.g. u123.csv -> u123."""
    return file_path.stem


def load_user_csv_files(file_paths):
    """Load multiple EDNet-KT1 user CSV files and attach user_id from filename."""
    frames = []

    for file_path in file_paths:
        user_df = pd.read_csv(file_path)
        user_df["user_id"] = extract_user_id_from_filename(file_path)
        frames.append(user_df)

    if len(frames) == 0:
        return pd.DataFrame()

    merged_df = pd.concat(frames, ignore_index=True)

    merged_df["timestamp"] = pd.to_datetime(
        merged_df["timestamp"],
        unit="ms",
        errors="coerce"
    )

    return merged_df


interactions_df = load_user_csv_files(selected_user_files)

print("Combined interactions shape:", interactions_df.shape)
print("Columns:")
print(list(interactions_df.columns))

display(interactions_df.head())


Combined interactions shape: (30819, 6)
Columns:
['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time', 'user_id']


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id
0,2019-08-06 12:56:30.868,1,q5012,b,38000,u1
1,2019-08-06 12:57:01.062,2,q4706,c,24000,u1
2,2019-08-06 12:58:13.432,3,q4366,b,68000,u1
3,2019-08-06 12:58:59.668,4,q4829,a,42000,u1
4,2019-08-06 13:00:01.774,5,q6528,b,59000,u1


In [18]:
# Basic interaction statistics.
print("Number of users:", interactions_df["user_id"].nunique())
print("Number of interactions:", len(interactions_df))
print("Number of unique questions:", interactions_df["question_id"].nunique())

print("Interactions per user:")
display(interactions_df.groupby("user_id").size().describe())


Number of users: 100
Number of interactions: 30819
Number of unique questions: 8356
Interactions per user:


count     100.000000
mean      308.190000
std       670.533749
min         1.000000
25%         7.750000
50%        30.000000
75%       107.500000
max      3436.000000
dtype: float64

In [19]:
# Clean question metadata.
questions_clean_df = questions_df.copy()

# Ensure question_id has the same type in both dataframes.
interactions_df["question_id"] = interactions_df["question_id"].astype(str)
questions_clean_df["question_id"] = questions_clean_df["question_id"].astype(str)

# Keep only the columns needed for Phase 1.
questions_clean_df = questions_clean_df[
    ["question_id", "correct_answer", "tags"]
].copy()

print("Clean question metadata shape:", questions_clean_df.shape)
display(questions_clean_df.head())


Clean question metadata shape: (13169, 3)


,question_id,correct_answer,tags
0,q1,b,1;2;179;181
1,q2,a,15;2;182
2,q3,b,14;2;179;183
3,q4,b,9;2;179;184
4,q5,c,8;2;179;181


In [21]:
# Join user interactions to question metadata.
rows_before = len(interactions_df)

kt_df = interactions_df.merge(
    questions_clean_df,
    on="question_id",
    how="left",
    validate="many_to_one"
)

print("KT dataframe shape:", kt_df.shape)
print("Rows before join:", rows_before)
print("Rows after join:", len(kt_df))

unmatched = kt_df["correct_answer"].isna().sum()

if unmatched > 0:
    print(
        f"WARNING: {unmatched:,} interactions "
        f"({unmatched / rows_before:.1%}) had no matching question_id "
        "in questions.csv and will be dropped."
    )

kt_df = kt_df.dropna(subset=["correct_answer", "tags", "user_id"]).copy()

print(f"Rows retained after metadata join: {len(kt_df):,} of {rows_before:,}")
display(kt_df.head())


KT dataframe shape: (30819, 8)
Rows before join: 30819
Rows after join: 30819
Rows retained after metadata join: 30,819 of 30,819


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id,correct_answer,tags
0,2019-08-06 12:56:30.868,1,q5012,b,38000,u1,c,74
1,2019-08-06 12:57:01.062,2,q4706,c,24000,u1,c,71
2,2019-08-06 12:58:13.432,3,q4366,b,68000,u1,b,103
3,2019-08-06 12:58:59.668,4,q4829,a,42000,u1,c,83
4,2019-08-06 13:00:01.774,5,q6528,b,59000,u1,d,90


In [22]:
# Create correctness label.

kt_df["is_correct"] = (
    kt_df["user_answer"].astype(str).str.strip().str.lower()
    ==
    kt_df["correct_answer"].astype(str).str.strip().str.lower()
).astype(int)

print("Correctness distribution:")
display(kt_df["is_correct"].value_counts(normalize=True).rename("proportion"))

display(
    kt_df[["user_id", "question_id", "user_answer", "correct_answer", "is_correct"]].head()
)


Correctness distribution:


is_correct
1    0.664558
0    0.335442
Name: proportion, dtype: float64

,user_id,question_id,user_answer,correct_answer,is_correct
0,u1,q5012,b,c,0
1,u1,q4706,c,c,1
2,u1,q4366,b,b,1
3,u1,q4829,a,c,0
4,u1,q6528,b,d,0


In [23]:
# Create skill_name from the first EDNet tag.

def extract_first_tag(tags):
    """
    Extract the first skill tag from EDNet's tags field.
    For this Phase 1 smoke test, we use the first tag as the primary skill.
    """
    if pd.isna(tags):
        return "unknown_skill"

    tags = str(tags).strip()

    if tags == "":
        return "unknown_skill"

    if ";" in tags:
        return tags.split(";")[0].strip()

    return tags.split()[0].strip()


kt_df["skill_name"] = kt_df["tags"].apply(extract_first_tag)

print("Number of unique skills:", kt_df["skill_name"].nunique())
print("Top 10 skills by interaction count:")
display(kt_df["skill_name"].value_counts().head(10))


Number of unique skills: 141
Top 10 skills by interaction count:


skill_name
179    1694
24     1363
30      974
85      912
52      794
77      773
76      669
83      646
53      622
54      555
Name: count, dtype: int64

In [24]:
print("Data pipeline ready:")
print(f"  Interactions     : {len(kt_df):,}")
print(f"  Unique users     : {kt_df['user_id'].nunique():,}")
print(f"  Unique questions : {kt_df['question_id'].nunique():,}")
print(f"  Unique skills    : {kt_df['skill_name'].nunique():,}")
print(f"  Overall accuracy : {kt_df['is_correct'].mean():.2%}")

display(
    kt_df[["user_id", "question_id", "skill_name", "is_correct"]].head()
)


Data pipeline ready:
  Interactions     : 30,819
  Unique users     : 100
  Unique questions : 8,356
  Unique skills    : 141
  Overall accuracy : 66.46%


,user_id,question_id,skill_name,is_correct
0,u1,q5012,74,0
1,u1,q4706,71,1
2,u1,q4366,103,1
3,u1,q4829,83,0
4,u1,q6528,90,0


In [25]:
# Sort interactions by user and timestamp.
kt_df = kt_df.sort_values(
    by=["user_id", "timestamp", "solving_id"],
    ascending=True
).reset_index(drop=True)

# Sequence index within each user.
kt_df["order_id"] = kt_df.groupby("user_id").cumcount() + 1

display(kt_df.head())


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id,correct_answer,tags,is_correct,skill_name,order_id
0,2019-08-06 12:56:30.868,1,q5012,b,38000,u1,c,74,0,74,1
1,2019-08-06 12:57:01.062,2,q4706,c,24000,u1,c,71,1,71,2
2,2019-08-06 12:58:13.432,3,q4366,b,68000,u1,b,103,1,103,3
3,2019-08-06 12:58:59.668,4,q4829,a,42000,u1,c,83,0,83,4
4,2019-08-06 13:00:01.774,5,q6528,b,59000,u1,d,90,0,90,5


In [26]:
# Create student-level train/test split.
#The split is done at user level to avoid the same learner appearing in both train and test.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=1 - CONFIG["train_size"],
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(
        kt_df,
        groups=kt_df["user_id"]
    )
)

train_raw_df = kt_df.iloc[train_idx].copy()
test_raw_df = kt_df.iloc[test_idx].copy()

print("Before skill filtering:")
print("Train shape:", train_raw_df.shape)
print("Test shape:", test_raw_df.shape)
print("Train users:", train_raw_df["user_id"].nunique())
print("Test users:", test_raw_df["user_id"].nunique())

overlap_users = set(train_raw_df["user_id"]) & set(test_raw_df["user_id"])
print("User overlap between train and test:", len(overlap_users))


Before skill filtering:
Train shape: (22388, 11)
Test shape: (8431, 11)
Train users: 80
Test users: 20
User overlap between train and test: 0


In [27]:
# Prepare BKT train/test dataframes.
bkt_train = train_raw_df[["user_id", "skill_name", "is_correct"]].copy()
bkt_test = test_raw_df[["user_id", "skill_name", "is_correct"]].copy()

bkt_train = bkt_train.rename(columns={"is_correct": "correct"})
bkt_test = bkt_test.rename(columns={"is_correct": "correct"})

bkt_train["user_id"] = bkt_train["user_id"].astype(str)
bkt_train["skill_name"] = bkt_train["skill_name"].astype(str)
bkt_train["correct"] = bkt_train["correct"].astype(int)

bkt_test["user_id"] = bkt_test["user_id"].astype(str)
bkt_test["skill_name"] = bkt_test["skill_name"].astype(str)
bkt_test["correct"] = bkt_test["correct"].astype(int)

print("BKT train shape before filtering:", bkt_train.shape)
print("BKT test shape before filtering:", bkt_test.shape)


BKT train shape before filtering: (22388, 3)
BKT test shape before filtering: (8431, 3)


In [28]:
MIN_INTERACTIONS_PER_SKILL = CONFIG["min_interactions_per_skill"]

skill_counts_train = bkt_train.groupby("skill_name")["correct"].count()
valid_skills = skill_counts_train[
    skill_counts_train >= MIN_INTERACTIONS_PER_SKILL
].index

bkt_train = bkt_train[bkt_train["skill_name"].isin(valid_skills)].copy()
bkt_test_seen_skills = bkt_test[bkt_test["skill_name"].isin(valid_skills)].copy()

print(f"Skills retained (>= {MIN_INTERACTIONS_PER_SKILL} training observations): {len(valid_skills):,}")
print(f"Train interactions: {len(bkt_train):,}")
print(f"Test interactions after skill filtering: {len(bkt_test_seen_skills):,}")
print(f"Train users: {bkt_train['user_id'].nunique():,}")
print(f"Test users: {bkt_test_seen_skills['user_id'].nunique():,}")


Skills retained (>= 10 training observations): 137
Train interactions: 22,362
Test interactions after skill filtering: 8,423
Train users: 80
Test users: 20


In [29]:
# Check train/test distributions.
print("Train correctness distribution:")
display(bkt_train["correct"].value_counts(normalize=True).rename("proportion"))

print("Test correctness distribution:")
display(bkt_test_seen_skills["correct"].value_counts(normalize=True).rename("proportion"))

print("Train skills:", bkt_train["skill_name"].nunique())
print("Test skills evaluated:", bkt_test_seen_skills["skill_name"].nunique())


Train correctness distribution:


correct
1    0.655353
0    0.344647
Name: proportion, dtype: float64

Test correctness distribution:


correct
1    0.688828
0    0.311172
Name: proportion, dtype: float64

Train skills: 137
Test skills evaluated: 136


In [31]:
import sys
!{sys.executable} -m pip install -q pyBKT

from pyBKT.models import Model as BKTModel



You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [32]:
# Train BKT model.
bkt_model = BKTModel(seed=SEED, num_fits=1)
bkt_model.fit(data=bkt_train)

print("BKT model training complete.")


BKT model training complete.


In [33]:
# Generate BKT predictions on test rows whose skills were retained from training.
bkt_predictions = bkt_model.predict(data=bkt_test_seen_skills)

print("Prediction dataframe shape:", bkt_predictions.shape)
print("Prediction columns:")
print(list(bkt_predictions.columns))

display(bkt_predictions.head())


Prediction dataframe shape: (8423, 5)
Prediction columns:
['user_id', 'skill_name', 'correct', 'correct_predictions', 'state_predictions']


,user_id,skill_name,correct,correct_predictions,state_predictions
0,u1,74,0,NaN,NaN
1,u1,71,1,NaN,NaN
2,u1,103,1,NaN,NaN
3,u1,83,0,NaN,NaN
4,u1,90,0,NaN,NaN


In [34]:
# Manually calculate BKT AUC and accuracy, safely handling NaNs.
possible_prediction_columns = [
    "correct_predictions",
    "correct_prediction",
    "prediction",
    "predictions",
    "probability",
]

prediction_column = None

for col in possible_prediction_columns:
    if col in bkt_predictions.columns:
        prediction_column = col
        break

if prediction_column is None:
    raise ValueError(
        "Could not automatically find the prediction column. "
        f"Available columns: {list(bkt_predictions.columns)}"
    )

print(f"Using prediction column: {prediction_column}")

eval_df = bkt_predictions[["correct", prediction_column]].copy()

print("Rows before removing NaNs:", len(eval_df))
print("NaN predictions:", eval_df[prediction_column].isna().sum())

eval_df = eval_df.dropna(subset=["correct", prediction_column])

print("Rows after removing NaNs:", len(eval_df))

if len(eval_df) == 0:
    raise ValueError("No rows left after removing NaN predictions.")

y_true = eval_df["correct"].astype(int)
y_score = eval_df[prediction_column].astype(float)

print("Correct label distribution after NaN removal:")
print(y_true.value_counts())

if y_true.nunique() < 2:
    bkt_auc = None
    print("AUC cannot be calculated because y_true has only one class.")
else:
    bkt_auc = roc_auc_score(y_true, y_score)
    print(f"BKT Test AUC: {bkt_auc:.4f}")

bkt_acc = accuracy_score(y_true, (y_score >= 0.5).astype(int))
print(f"BKT Test ACC: {bkt_acc:.4f}")


Using prediction column: correct_predictions
Rows before removing NaNs: 8423
NaN predictions: 783
Rows after removing NaNs: 7640
Correct label distribution after NaN removal:
correct
1    5341
0    2299
Name: count, dtype: int64
BKT Test AUC: 0.5000
BKT Test ACC: 0.3009


In [36]:
print("Learned BKT parameters, first 10 rows:")

try:
    display(bkt_model.params().head(20))
except Exception as e:
    print("Could not display BKT parameters.")
    print(e)

Learned BKT parameters, first 10 rows:


value
skill param   class          
84    prior   default     NaN
      learns  default 1.00000
      guesses default 0.50000
      slips   default 0.50000
      forgets default 0.00000
34    prior   default     NaN
      learns  default 1.00000
      guesses default 0.50000
      slips   default 0.50000
      forgets default 0.00000
72    prior   default     NaN
      learns  default 1.00000
      guesses default 0.50000
      slips   default 0.50000
      forgets default 0.00000
74    prior   default     NaN
      learns  default 1.00000
      guesses default 0.50000
      slips   default 0.50000
      forgets default 0.00000

In [37]:
phase1_bkt_summary = pd.DataFrame([
    {
        "model": "BKT",
        "n_users_total": kt_df["user_id"].nunique(),
        "n_train_users": bkt_train["user_id"].nunique(),
        "n_test_users_original": bkt_test["user_id"].nunique(),
        "n_test_users_evaluated": bkt_test_seen_skills["user_id"].nunique(),
        "n_interactions_total": len(kt_df),
        "n_train_interactions": len(bkt_train),
        "n_test_interactions_original": len(bkt_test),
        "n_test_interactions_evaluated": len(bkt_test_seen_skills),
        "n_skills_total": kt_df["skill_name"].nunique(),
        "n_skills_train": bkt_train["skill_name"].nunique(),
        "n_skills_test_original": bkt_test["skill_name"].nunique(),
        "n_skills_test_evaluated": bkt_test_seen_skills["skill_name"].nunique(),
        "auc": bkt_auc,
        "accuracy": bkt_acc,
        "notes": "Phase 1 BKT smoke test; rare-skill filtering fitted on training data only."
    }
])

display(phase1_bkt_summary)


,model,n_users_total,n_train_users,n_test_users_original,n_test_users_evaluated,n_interactions_total,n_train_interactions,n_test_interactions_original,n_test_interactions_evaluated,n_skills_total,n_skills_train,n_skills_test_original,n_skills_test_evaluated,auc,accuracy,notes
0,BKT,100,80,20,20,30819,22362,8431,8423,141,137,140,136,0.50000,0.30092,Phase 1 BKT smoke test; rare-skill filtering f...
